# nlm_synth quickstart

This notebook walks through the core idea of the package: separate a landscape's
**spatial structure** from its **value distribution**, then measure how the
statistics of the combined field change as you coarsen the pixel size.

Sections:
1. Define a target NDVI distribution
2. Generate fields with different spatial structure but the same distribution
3. Confirm the distribution is reproduced exactly
4. Run the multi-scale Monte Carlo experiment
5. Fit Perlin parameters to an observed raster

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import nlm_synth as ns

print("nlm_synth", ns.__version__)

## 1. A target NDVI distribution

Any 1-D array of values works. Here we build a bimodal sample standing in for a
scene with a vegetated mode and a bare-soil mode. To use real data instead, load
your own pixels:

```python
import rioxarray as rxr
samples = rxr.open_rasterio('scene.tif', masked=True).values.ravel()
samples = samples[np.isfinite(samples)]
```

In [ ]:
rng = np.random.default_rng(123)
vegetation = rng.normal(0.70, 0.08, size=50_000)
soil = rng.normal(0.20, 0.09, size=30_000)
samples = np.clip(np.hstack([vegetation, soil]), -0.2, 1.0)

fig, _ = ns.plot_marginal(samples, label="NDVI")
plt.show()

## 2. Same distribution, different spatial structure

`synth_ndvi_from_distribution` generates a neutral landscape model and quantile-maps
it onto the sample distribution. The Perlin parameters control the structure:

| parameter | effect |
| --- | --- |
| `periods` | patch size: more periods means finer patches |
| `octaves` | number of noise layers summed; more octaves adds fine detail |
| `lacunarity` | how fast frequency grows between octaves |
| `persistence` | how fast amplitude decays between octaves; higher keeps more fine detail |

In [ ]:
configs = [
    ("low frequency", dict(periods=(2, 2), octaves=3, lacunarity=2, persistence=0.7)),
    ("mid frequency", dict(periods=(4, 4), octaves=5, lacunarity=2, persistence=0.6)),
    ("high frequency", dict(periods=(8, 8), octaves=6, lacunarity=2, persistence=0.5)),
]

fields = [
    ns.synth_ndvi_from_distribution(256, 256, samples, method="perlin",
                                    method_kwargs=kwargs, seed=1 + i)
    for i, (_, kwargs) in enumerate(configs)
]

titles = [f"{name}\nMoran's I = {ns.morans_i(field):.3f}"
          for (name, _), field in zip(configs, fields, strict=True)]
fig, _ = ns.plot_field_grid(fields, titles=titles, ncols=3)
plt.show()

## 3. The marginal distribution is preserved exactly

All three fields look different but have the *same* value distribution, because
the mapping is by rank. This is what makes the comparison in section 4 clean:
any difference in how the statistics behave with scale is attributable to
spatial structure alone, not to the values themselves.

In [ ]:
print(f"{'quantile':>9}  {'target':>8}" + "".join(f"  {name:>15}" for name, _ in configs))
for q in (1, 5, 25, 50, 75, 95, 99):
    row = f"{q:>8}%  {np.percentile(samples, q):8.4f}"
    row += "".join(f"  {np.percentile(field, q):15.4f}" for field in fields)
    print(row)

## 4. How statistics change with pixel size

`run_experiments` synthesises many realisations per generator, block-averages each
to a range of pixel sizes, and records summary statistics at every scale.

The result is the package's central claim: **spatial structure determines how
quickly landscape heterogeneity disappears as pixels grow.** Coarse-grained
structure retains autocorrelation far longer than fine-grained structure, even
though both start from an identical distribution.

In [ ]:
grid = [
    {"label": "perlin_LF", "method": "perlin",
     "method_kwargs": dict(periods=(2, 2), octaves=3, lacunarity=2, persistence=0.7)},
    {"label": "perlin_HF", "method": "perlin",
     "method_kwargs": dict(periods=(8, 8), octaves=6, lacunarity=2, persistence=0.5)},
    {"label": "cluster_nn", "method": "cluster",
     "method_kwargs": dict(p=0.55, cluster_p=0.65, periods=(6, 6),
                           octaves=2, lacunarity=2, persistence=0.4)},
]

df, meta = ns.run_experiments(
    samples, nrow=256, ncol=256, generator_grid=grid,
    coarsen_factors=(1, 2, 4, 8, 16, 32), n_runs=5, random_seed=44, progress=True,
)
df.head()

In [ ]:
df.pivot_table(index="factor", columns="label", values="morans_I").round(3)

In [ ]:
for metric in ("morans_I", "variance"):
    fig, _ = ns.plot_metric_by_scale(df, metric=metric)
    plt.show()

The mean is flat across scales (block averaging preserves it), while variance and
Moran's I both fall — but at rates that depend entirely on the generator. That
gap is the scale effect this package exists to quantify.

In [ ]:
df.groupby(["label", "factor"])[["mean", "variance", "morans_I"]].mean().round(3)

## 5. Fitting Perlin parameters to a real raster

Given an observed scene, `fit_perlin_parameters_geotiff` grid-searches for the
Perlin parameters whose structure best matches it, comparing the radially
averaged power spectrum and Moran's I of the rank-transformed images. Because
the comparison is on ranks, the fit is unaffected by the scene's value
distribution.

Here we synthesise a scene with known parameters and check that they are
recovered. Point `in_tif` at your own file to use real data.

In [ ]:
from rasterio.transform import from_origin

from nlm_synth.approximations import fit_perlin_parameters_geotiff
from nlm_synth.geox import write_geotiff

truth = dict(periods=(8, 8), octaves=4, lacunarity=2, persistence=0.5)
scene = ns.rank_map_to_distribution(
    ns.perlin_field(200, 200, seed=42, **truth), samples
)
write_geotiff("scene.tif", scene, from_origin(500_000, 4_000_000, 30, 30), "EPSG:32611")

best = fit_perlin_parameters_geotiff("scene.tif", verbose=False)
print("truth:", truth)
print("fit:  ", {k: best[k] for k in ("periods", "octaves", "lacunarity", "persistence")})
print(f"Moran's I  target={best['target_moran']:.4f}  fitted={best['moran']:.4f}")

Here the search recovers the generating parameters exactly. That is not guaranteed:
Perlin parameters are **not identifiable**, because different combinations of `periods`,
`octaves`, `lacunarity` and `persistence` can produce near-identical spectra. On real
imagery the fit often lands on a different parameter set that matches the observed
structure just as well — compare the two Moran's I values rather than the parameter
lists. Treat a fit as *a* parameter set reproducing the observed structure, not *the*
true one.

In [ ]:
fitted = ns.rank_map_to_distribution(
    ns.perlin_field(
        *scene.shape, periods=best["periods"], octaves=best["octaves"],
        lacunarity=best["lacunarity"], persistence=best["persistence"], seed=7,
    ),
    samples,
)

fig, _ = ns.plot_field_grid(
    [scene, fitted],
    titles=[f"observed (I={ns.morans_i(scene):.3f})",
            f"synthetic (I={ns.morans_i(fitted):.3f})"],
    ncols=2,
)
plt.show()

## Where to go next

- `examples/run_monte_carlo.py` — the full experiment as a script
- `examples/run_geotiff_monte_carlo.py` — same, writing georeferenced GeoTIFFs
- `examples/fit_perlin_to_raster.py` — fit parameters to your own scene
- `nlm-synth --help` — the same workflows from the command line